In [ ]:
import numpy as np
import xarray as xr
import warnings
from pandas import date_range
from pathlib import Path

from grid_toolbox.basic_latlon import get_cells_area

import pycompo.core.composite as pccompo
import pycompo.core.coord as pccoord
import pycompo.core.ellipse as pcellipse
import pycompo.core.feature_cutout as pcfeatcut
import pycompo.core.filter as pcfilter
import pycompo.core.plot as pcplot
import pycompo.core.sigtest as pcsig
import pycompo.core.sst_features as pcsst
import pycompo.core.wind as pcwind
import pycompo.core.utils as pcutil

warnings.filterwarnings(action='ignore')

# read in configuration file
config_file = "/home/m/m300738/libs/pycompo/config/settings_template.yaml"
config = pcutil.read_yaml_config(config_file)

start_time = config['data']['analysis_time'][0]
end_time = config['data']['analysis_time'][1]
analysis_times = [
    np.datetime64(t) for t in date_range(
        np.datetime64(start_time), np.datetime64(end_time), freq='MS',
        )
    ]
feature_var = config['data']['feature_var']
varlist = [feature_var] + config['data']['wind_vars'] + \
    config['data']['study_vars']

### Read in data, building SST anomalies and their gradients using Gaussian filter

In [2]:
infiles = []
for var in varlist:
    inpath = Path(config['data']['inpaths'][var])
    in_pattern = f"{config['exp']}_tropical_{var}_*.nc"
    infiles.extend(sorted([str(f) for f in inpath.rglob(in_pattern)]))
dset = xr.open_mfdataset(infiles, parallel=False).squeeze()
dset = pcutil.subsample_data(dset, start_time, end_time, config)
if config['test']: dset = dset.isel(time=slice(20, 22))
if 'height_2' in dset.coords: dset = dset.drop('height_2')

# Detrend data with climatology
if config['detrend']['switch']:
    dset, feature_var, varlist = pcfilter.detrend_with_hourly_climatology(
        dset, feature_var, config,
        )

for var in varlist: dset[var] = dset[var].compute()

# scale separation
filter_vars = [feature_var] + config['data']['wind_vars']
if config['composite']['type'] == 'anomaly':
    filter_vars += config['data']['study_vars']
dset_filter = pcfilter.get_gaussian_filter_bg_ano(
    dset[filter_vars], **config['filter']
    )

if config['composite']['type'] == 'anomaly':
    merge_dsets = [
        dset_filter[[f"{var}_bg" for var in config['data']['wind_vars']]],
        dset_filter[[f"{var}_ano" for var in filter_vars]],
        ]
    grad_var = f"{feature_var}_ano"
elif config['composite']['type'] == 'absolute':
    merge_dsets = [
        dset_filter[[f"{var}_bg" for var in config['data']['wind_vars']]],
        dset_filter[f"{feature_var}_ano"],
        dset,
        ]
    grad_var = feature_var

# calculate stability proxy if input variables are available
if 'ts_bg' in dset_filter and 'tas_bg' in dset_filter:
    dset_filter['tas-ts_bg'] = dset_filter['tas_bg'] - dset_filter['ts_bg']
    merge_dsets.append(dset_filter['tas-ts_bg'])
if 'sfc_rho_bg' in dset_filter:
    merge_dsets.append(dset_filter['sfc_rho_bg'])
if 'pr_bg' in dset_filter:
    merge_dsets.append(dset_filter['pr_bg'])
    dset_filter['pr_fraction'] = dset_filter['pr_ano']/dset_filter['pr_bg']
    merge_dsets.append(dset_filter['pr_fraction'])

dset = xr.merge(merge_dsets)

# add timelag, calculate gradients, and subsampling
dset = pcutil.add_timelag_idx_space(
    dset, f"{feature_var}_ano", config['data']['timelag_idx'],
    )
dset = pccoord.calc_sphere_gradient_laplacian(dset, grad_var)
dset = pccoord.calc_sphere_convergence_components(
    dset, tuple(f"{var}_ano" for var in config['data']['wind_vars']),
    )

# calculate optional quantities
if 'tas' in config['data']['study_vars']:
    if config['composite']['type'] == 'anomaly':
        dset = pccoord.calc_sphere_gradient_laplacian(dset, 'tas_ano')
    elif config['composite']['type'] == 'absolute':
        dset = pccoord.calc_sphere_gradient_laplacian(dset, 'tas')

if 'ps' in config['data']['study_vars']:
    if config['composite']['type'] == 'anomaly':
        dset = pccoord.calc_sphere_laplacian(dset, 'ps_ano')
    elif config['composite']['type'] == 'absolute':
        dset = pccoord.calc_sphere_laplacian(dset, 'ps')        

# subsampling to latitudinal belt of interest
dset['cell_area'] = get_cells_area(dset)
dset = dset.sel(lat=slice(*config['lat_range']), drop=True)

# calculate population mean for correct Null hypothesis in significance testing
popmeans_alltrops = pcsig.calc_popmeans(dset)
popmeans_alltrops = popmeans_alltrops.mean(dim='time')

if config['composite']['rainbelt_subsampling']['switch']:
    rainbelt = pccompo.get_rainbelt(analysis_times, config, quantile=0.8)
    if config['test']: rainbelt = rainbelt.isel(time=slice(20, 22))
    rainbelt = rainbelt.compute()
    popmeans_rainbelt = pcsig.calc_popmeans(dset.where(rainbelt))
    popmeans_rainbelt = popmeans_rainbelt.mean(dim='time')

orig_coords = pccoord.get_coords_orig(dset.drop('time'))

### Detection of SST clusters and cutout of corresponding data

In [ ]:
dset[f"{feature_var}_feature"], feature_props = pcsst.extract_sst_features(
    dset[f"{feature_var}_ano_detect"], **config['feature'],
    )
feature_props["time"] = feature_props["time"].roll(
    feature=feature_props.dims['feature']//2, roll_coords=False,
    )
feature_props, feature_data = pcfeatcut.get_featcen_data_cutouts(
    dset, feature_props, config['cutout']['search_RadRatio'],
    )
feature_props = pcwind.calc_feature_bg_wind(
    feature_props, feature_data, config['data']['wind_vars'],
    )
feature_props, feature_data = pcsst.add_more_feature_props(
    feature_props, feature_data, orig_coords,
    )

### Creation of composites

In [ ]:
# create feature ellipse and filter with respect to aspect ratio
feature_ellipse = pcellipse.get_ellipse_params(feature_props, orig_coords)
feature_data, feature_props, feature_ellipse = pcsst.filter_asprat(
    feature_data, feature_props, feature_ellipse,
    max_asprat=config['feature']['max_ar'],
    )

# coordinate transformation
feature_data = pccoord.add_featcen_coords(
    orig_coords, feature_data, feature_props, feature_ellipse,
    )
feature_data = pcwind.add_wind_grads(feature_data, feature_props, grad_var)
if 'tas' in config['data']['study_vars']:
    if config['composite']['type'] == 'anomaly':
        grad_var2 = "tas_ano"
    elif config['composite']['type'] == 'absolute':
        grad_var2 = "tas"
feature_data = pcwind.add_wind_grads(feature_data, feature_props, grad_var2)
feature_data = pcwind.add_rotate_winds(feature_data, feature_props)
feature_compo_data = pccompo.get_compo_coords_ds(feature_data, config)

feature_props = feature_props.where(
        feature_props['feature_id'].isin(feature_compo_data['feature_id']),
        drop=True,
    )

# Significance tests
_, pvalue = pcsig.calc_compo_ttest(
    feature_compo_data, popmean=popmeans_alltrops,
    )
local_significane = pcsig.get_local_significance(pvalue, alpha=0.05)
field_significane = pcsig.get_field_significance(pvalue, alpha_FDR=0.1)